# Touchless Media Control — Jetson Nano HCI System

**Objective:** Real-time hand-gesture recognition on an NVIDIA Jetson Nano that translates gestures into media-control commands for VLC.

**Stack:** MediaPipe Hands (`hand_landmarker.task`) + OpenCV + pynput / xdotool

**Performance Targets:**
| Metric | Target |
|---|---|
| Gesture accuracy | > 90 % (controlled lighting) |
| End-to-end latency | < 200 ms |
| Frame rate | ≥ 15 FPS |

## 1. Environment Check
Verify Python version, CUDA availability, and required packages before running anything else.

In [ ]:
import sys, importlib, shutil, os

print(f"Python  : {sys.version.split()[0]}")
print(f"Platform: {sys.platform}")

def ver(pkg):
    try:
        m = importlib.import_module(pkg)
        return getattr(m, '__version__', 'installed (version unknown)')
    except Exception:
        return '** NOT INSTALLED **'

print(f"numpy      : {ver('numpy')}")
print(f"cv2        : {ver('cv2')}")
print(f"mediapipe  : {ver('mediapipe')}")
print(f"protobuf   : {ver('google.protobuf')}")
print(f"pynput     : {ver('pynput')}")

# Check for the hand-landmarker model
MODEL_PATH = os.path.join(os.getcwd(), 'hand_landmarker.task')
print(f"\nModel file : {'FOUND' if os.path.isfile(MODEL_PATH) else 'MISSING'} — {MODEL_PATH}")

# CUDA quick check (torch is optional on Jetson; CUDA may be used via OpenCV DNN)
try:
    import torch
    print(f"torch      : {torch.__version__}  CUDA={torch.cuda.is_available()}")
except ImportError:
    print("torch      : not installed (optional)")


## 2. Gesture Set & VLC Mapping

Each gesture is encoded as a **5-bit finger pattern** `[Thumb, Index, Middle, Ring, Pinky]` (1 = up, 0 = down).

| # | Gesture | Pattern | VLC Action | Shortcut |
|---|---------|---------|------------|----------|
| 1 | Open Palm | `11111` | Play / Pause | `Space` |
| 2 | Fist | `00000` | Mute / Unmute | `M` |
| 3 | Thumbs Up | `10000` | Volume Up | `Ctrl + Up` |
| 4 | Pinky Only | `00001` | Volume Down | `Ctrl + Down` |
| 5 | Peace / V | `01100` | Next Track | `N` |
| 6 | Index Point | `01000` | Previous Track | `P` |
| 7 | Three (I-M-R) | `01110` | Fullscreen Toggle | `F` |

A **cooldown timer** prevents the same action from firing repeatedly.

In [ ]:
import cv2
import time
import numpy as np
import mediapipe as mp

# ──────────────────────────────────────────────
# Configuration
# ──────────────────────────────────────────────
CAMERA_SRC   = 0          # webcam index (change for CSI or USB-2)
FRAME_W      = 640
FRAME_H      = 480
MAX_HANDS    = 1           # single-hand control is faster & avoids ambiguity
COOLDOWN_SEC = 1.0         # minimum seconds between repeated same-action fires
HOLD_FRAMES  = 3           # gesture must be stable for N consecutive frames

MODEL_TASK   = 'hand_landmarker.task'   # MediaPipe Tasks model in project folder

# Gesture pattern → (action_name, description)
GESTURE_MAP = {
    (1,1,1,1,1): ('play_pause',  'Play / Pause'),
    (0,0,0,0,0): ('mute',        'Mute / Unmute'),
    (1,0,0,0,0): ('vol_up',      'Volume Up'),
    (0,0,0,0,1): ('vol_down',    'Volume Down'),
    (0,1,1,0,0): ('next_track',  'Next Track'),
    (0,1,0,0,0): ('prev_track',  'Previous Track'),
    (0,1,1,1,0): ('fullscreen',  'Fullscreen Toggle'),
}

print(f"Defined {len(GESTURE_MAP)} gestures")


## 3. Hand-Landmark Detection

Uses the provided `hand_landmarker.task` model via the **MediaPipe Tasks** API.  
Falls back to the legacy `mediapipe.solutions.hands` if the Tasks API is unavailable on the installed version.

In [ ]:
# ──────────────────────────────────────────────
# Initialise MediaPipe hand detector
# ──────────────────────────────────────────────
USE_TASKS_API = False       # will be set True if Tasks API loads successfully

try:
    from mediapipe.tasks import python as mp_tasks
    from mediapipe.tasks.python import vision as mp_vision

    base_opts = mp_tasks.BaseOptions(model_asset_path=MODEL_TASK)
    hand_opts = mp_vision.HandLandmarkerOptions(
        base_options=base_opts,
        num_hands=MAX_HANDS,
        min_hand_detection_confidence=0.5,
        min_hand_presence_confidence=0.5,
        min_tracking_confidence=0.5,
        running_mode=mp_vision.RunningMode.VIDEO,
    )
    hand_detector = mp_vision.HandLandmarker.create_from_options(hand_opts)
    USE_TASKS_API = True
    print("✓ MediaPipe Tasks API loaded with hand_landmarker.task")
except Exception as e:
    print(f"Tasks API unavailable ({e}); falling back to mediapipe.solutions.hands")
    mp_hands_mod = mp.solutions.hands
    hand_detector = mp_hands_mod.Hands(
        static_image_mode=False,
        max_num_hands=MAX_HANDS,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )

mp_drawing  = mp.solutions.drawing_utils
mp_hand_con = mp.solutions.hands.HAND_CONNECTIONS
print(f"Using Tasks API: {USE_TASKS_API}")


## 4. Gesture Classification

`fingers_up()` → 5-element tuple per hand → lookup in `GESTURE_MAP`.  
A **stability buffer** ensures the same pattern is held for `HOLD_FRAMES` consecutive frames before triggering.

In [ ]:
# ──────────────────────────────────────────────
# Finger-state & gesture helpers
# ──────────────────────────────────────────────

def fingers_up(landmarks, handedness_label):
    """
    Return a 5-tuple of ints (0/1) representing
    [Thumb, Index, Middle, Ring, Pinky] up state.

    Works with both Tasks-API NormalizedLandmark list and
    legacy solutions LandmarkList.
    """
    # Normalise access: Tasks API gives list of NormalizedLandmark objects
    # Legacy API gives landmark_pb2 with .landmark attribute
    if hasattr(landmarks, 'landmark'):
        lm = landmarks.landmark          # legacy
    else:
        lm = landmarks                   # Tasks API (already a list)

    fingers = []

    # Thumb: compare x of tip(4) vs IP(3) — flipped for left/right
    if handedness_label == 'Right':
        fingers.append(int(lm[4].x < lm[3].x))
    else:
        fingers.append(int(lm[4].x > lm[3].x))

    # Index(8), Middle(12), Ring(16), Pinky(20): tip y < pip y  →  finger up
    for tip in (8, 12, 16, 20):
        fingers.append(int(lm[tip].y < lm[tip - 2].y))

    return tuple(fingers)


def classify_gesture(pattern):
    """Look up the finger pattern in GESTURE_MAP; return (action, label) or None."""
    return GESTURE_MAP.get(pattern)


class GestureStabiliser:
    """Require the same pattern for `hold` consecutive frames before firing."""

    def __init__(self, hold=HOLD_FRAMES, cooldown=COOLDOWN_SEC):
        self.hold = hold
        self.cooldown = cooldown
        self._prev_pattern = None
        self._streak = 0
        self._last_action_time = {}     # action_name → timestamp

    def update(self, pattern):
        """Feed a new frame's pattern; returns (action, label) if it should fire, else None."""
        if pattern == self._prev_pattern:
            self._streak += 1
        else:
            self._prev_pattern = pattern
            self._streak = 1

        if self._streak < self.hold:
            return None

        result = classify_gesture(pattern)
        if result is None:
            return None

        action, label = result
        now = time.time()
        if now - self._last_action_time.get(action, 0) < self.cooldown:
            return None                  # cooldown still active

        self._last_action_time[action] = now
        return (action, label)


stabiliser = GestureStabiliser()
print("✓ Gesture classifier & stabiliser ready")


## 5. Media Control — VLC Key Bindings

Maps each gesture action to a VLC keyboard shortcut.  
- **Linux / Jetson:** uses `xdotool` (subprocess) — works headlessly.  
- **Windows (dev):** uses `pynput.keyboard`.

In [ ]:
# ──────────────────────────────────────────────
# VLC media-control dispatcher
# ──────────────────────────────────────────────
import subprocess, platform

IS_LINUX = platform.system() == 'Linux'

def _send_key_xdotool(keys):
    """Send key combo via xdotool (Linux / Jetson)."""
    subprocess.Popen(['xdotool', 'key', keys],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def _send_key_pynput(key_sequence):
    """Send key combo via pynput (Windows / macOS dev)."""
    from pynput.keyboard import Controller, Key
    kb = Controller()
    # key_sequence is a list of pynput Key / str
    held = []
    for k in key_sequence:
        kb.press(k)
        held.append(k)
    for k in reversed(held):
        kb.release(k)


# ── Action → key mapping for each platform ──

def vlc_action(action_name):
    """Dispatch a VLC action by name. Returns True if dispatched."""
    if IS_LINUX:
        xdo_map = {
            'play_pause':  'space',
            'mute':        'm',
            'vol_up':      'ctrl+Up',
            'vol_down':    'ctrl+Down',
            'next_track':  'n',
            'prev_track':  'p',
            'fullscreen':  'f',
        }
        keys = xdo_map.get(action_name)
        if keys:
            _send_key_xdotool(keys)
            return True
    else:
        from pynput.keyboard import Key
        pynput_map = {
            'play_pause':  [' '],
            'mute':        ['m'],
            'vol_up':      [Key.ctrl_l, Key.up],
            'vol_down':    [Key.ctrl_l, Key.down],
            'next_track':  ['n'],
            'prev_track':  ['p'],
            'fullscreen':  ['f'],
        }
        seq = pynput_map.get(action_name)
        if seq:
            _send_key_pynput(seq)
            return True
    return False

print(f"✓ VLC controller ready  (backend: {'xdotool' if IS_LINUX else 'pynput'})")


## 6. Performance Tracker

Logs per-frame FPS, detection latency, and end-to-end latency so we can verify targets after a run.

In [ ]:
# ──────────────────────────────────────────────
# Performance metrics collector
# ──────────────────────────────────────────────

class PerfTracker:
    """Collect per-frame timing data for post-run analysis."""

    def __init__(self):
        self.fps_log = []               # instantaneous FPS
        self.detect_ms = []             # hand-detection time (ms)
        self.e2e_ms = []                # full pipeline latency (ms)
        self.gesture_hits = 0           # frames where a gesture was recognised
        self.total_frames = 0
        self._prev_time = None

    def tick_start(self):
        self._t0 = time.perf_counter()

    def tick_detect_done(self):
        self._t_det = time.perf_counter()
        self.detect_ms.append((self._t_det - self._t0) * 1000)

    def tick_end(self):
        now = time.perf_counter()
        self.e2e_ms.append((now - self._t0) * 1000)
        if self._prev_time is not None:
            dt = now - self._prev_time
            self.fps_log.append(1.0 / dt if dt > 0 else 0)
        self._prev_time = now
        self.total_frames += 1

    def record_gesture(self):
        self.gesture_hits += 1

    def summary(self):
        if not self.fps_log:
            return "No data collected."
        fps_arr = np.array(self.fps_log)
        det_arr = np.array(self.detect_ms)
        e2e_arr = np.array(self.e2e_ms)
        lines = [
            f"Frames analysed : {self.total_frames}",
            f"Gestures fired  : {self.gesture_hits}",
            f"FPS             : mean={fps_arr.mean():.1f}  min={fps_arr.min():.1f}  max={fps_arr.max():.1f}",
            f"Detection (ms)  : mean={det_arr.mean():.1f}  p95={np.percentile(det_arr,95):.1f}",
            f"End-to-end (ms) : mean={e2e_arr.mean():.1f}  p95={np.percentile(e2e_arr,95):.1f}",
        ]
        return '\n'.join(lines)

perf = PerfTracker()
print("✓ Performance tracker ready")


## 7. Main Pipeline

Capture → Detect → Classify → Dispatch → Draw OSD → Loop.

Press **q** or **ESC** to exit the live window.

In [ ]:
# ──────────────────────────────────────────────
# OSD drawing helpers
# ──────────────────────────────────────────────

def draw_hud(frame, fps, gesture_label, pattern, latency_ms):
    """Draw a translucent HUD overlay on the frame."""
    h, w = frame.shape[:2]

    # Background strip
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (w, 80), (30, 30, 30), -1)
    cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)

    # FPS (left)
    col_fps = (0, 255, 0) if fps >= 15 else (0, 165, 255)
    cv2.putText(frame, f"FPS: {fps:.0f}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, col_fps, 2)

    # Latency (centre-left)
    col_lat = (0, 255, 0) if latency_ms < 200 else (0, 0, 255)
    cv2.putText(frame, f"Latency: {latency_ms:.0f}ms", (180, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, col_lat, 2)

    # Gesture (right)
    if gesture_label:
        cv2.putText(frame, gesture_label, (w - 280, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)

    # Finger pattern bar (bottom of HUD)
    labels = ['T', 'I', 'M', 'R', 'P']
    if pattern:
        for i, (lab, val) in enumerate(zip(labels, pattern)):
            x = 10 + i * 50
            col = (0, 255, 0) if val else (80, 80, 80)
            cv2.putText(frame, f"{lab}:{val}", (x, 65),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, col, 2)


def draw_landmarks_on_frame(frame, hand_landmarks):
    """Draw 21-point hand skeleton; works for both API types."""
    if hasattr(hand_landmarks, 'landmark'):
        # Legacy API — pass directly
        mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hand_con)
    else:
        # Tasks API — reconstruct a NormalizedLandmarkList for drawing
        from mediapipe.framework.formats import landmark_pb2
        proto = landmark_pb2.NormalizedLandmarkList()
        for lm in hand_landmarks:
            proto.landmark.add(x=lm.x, y=lm.y, z=lm.z)
        mp_drawing.draw_landmarks(frame, proto, mp_hand_con)

print("✓ Drawing helpers defined")


In [ ]:
# ──────────────────────────────────────────────
# Main real-time loop
# ──────────────────────────────────────────────

def run_gesture_control(src=CAMERA_SRC, width=FRAME_W, height=FRAME_H):
    """Launch webcam, detect hands, classify gestures, send VLC commands."""
    global perf, stabiliser
    perf = PerfTracker()
    stabiliser = GestureStabiliser()

    cap = cv2.VideoCapture(src)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, width)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, height)

    if not cap.isOpened():
        print(f"ERROR: cannot open camera {src}")
        return

    print(f"Camera opened ({width}x{height}).  Press 'q' / ESC to quit.")
    frame_ts = 0          # monotonic frame counter for Tasks API VIDEO mode

    while True:
        perf.tick_start()

        ret, frame = cap.read()
        if not ret:
            print("Camera read failed — exiting.")
            break

        frame = cv2.flip(frame, 1)       # mirror view
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # ── Hand detection ──────────────────────
        pattern = None
        gesture_label = ''

        if USE_TASKS_API:
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            frame_ts += 33                # ~30 fps timestamp step (ms)
            result = hand_detector.detect_for_video(mp_image, frame_ts)
            perf.tick_detect_done()

            if result.hand_landmarks:
                for lms, handed in zip(result.hand_landmarks, result.handedness):
                    label = handed[0].category_name   # 'Left' / 'Right'
                    pattern = fingers_up(lms, label)
                    draw_landmarks_on_frame(frame, lms)

                    # Per-hand label near wrist
                    h, w = frame.shape[:2]
                    x = int(lms[0].x * w)
                    y = int(lms[0].y * h)
                    cv2.putText(frame, f"{label}", (x - 20, y - 20),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 0), 2)
        else:
            results = hand_detector.process(rgb)
            perf.tick_detect_done()

            if results.multi_hand_landmarks and results.multi_handedness:
                for lms, handed in zip(results.multi_hand_landmarks, results.multi_handedness):
                    label = handed.classification[0].label
                    pattern = fingers_up(lms, label)
                    draw_landmarks_on_frame(frame, lms)

                    h, w = frame.shape[:2]
                    x = int(lms.landmark[0].x * w)
                    y = int(lms.landmark[0].y * h)
                    cv2.putText(frame, f"{label}", (x - 20, y - 20),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 0), 2)

        # ── Gesture classification ──────────────
        if pattern is not None:
            fired = stabiliser.update(pattern)
            if fired:
                action_name, gesture_label = fired
                vlc_action(action_name)
                perf.record_gesture()
                print(f"  ▶ {gesture_label}  ({action_name})")
        else:
            stabiliser.update(None)

        # ── HUD ─────────────────────────────────
        perf.tick_end()
        fps = perf.fps_log[-1] if perf.fps_log else 0
        lat = perf.e2e_ms[-1] if perf.e2e_ms else 0
        draw_hud(frame, fps, gesture_label, pattern, lat)

        cv2.imshow('Gesture Media Control', frame)

        key = cv2.waitKey(1) & 0xFF
        if key in (27, ord('q')):
            break

    cap.release()
    cv2.destroyAllWindows()
    print("\n── Run complete ──")
    print(perf.summary())

print("✓ run_gesture_control() defined — run the next cell to start")


## 8. Launch

> **Before running:** make sure VLC (or another media player) is open and focused on the Jetson.  
> The cell below opens the webcam window. Press **q** or **ESC** to stop.

In [ ]:
# ▶ Start the live gesture-control loop
run_gesture_control(src=CAMERA_SRC, width=FRAME_W, height=FRAME_H)


## 9. Post-Run Performance Analysis

Run the cell below **after** closing the gesture window to see FPS / latency plots and verify the performance targets.

In [ ]:
# ──────────────────────────────────────────────
# Analyse collected metrics
# ──────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')           # safe backend for notebooks
import matplotlib.pyplot as plt

print(perf.summary())
print()

if perf.fps_log:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # FPS over time
    axes[0].plot(perf.fps_log, linewidth=0.8, color='tab:green')
    axes[0].axhline(15, color='red', linestyle='--', label='15 FPS target')
    axes[0].set_title('FPS over time')
    axes[0].set_xlabel('Frame')
    axes[0].set_ylabel('FPS')
    axes[0].legend()

    # Detection latency
    axes[1].plot(perf.detect_ms, linewidth=0.8, color='tab:blue')
    axes[1].axhline(200, color='red', linestyle='--', label='200 ms target')
    axes[1].set_title('Detection latency')
    axes[1].set_xlabel('Frame')
    axes[1].set_ylabel('ms')
    axes[1].legend()

    # End-to-end latency
    axes[2].plot(perf.e2e_ms, linewidth=0.8, color='tab:orange')
    axes[2].axhline(200, color='red', linestyle='--', label='200 ms target')
    axes[2].set_title('End-to-end latency')
    axes[2].set_xlabel('Frame')
    axes[2].set_ylabel('ms')
    axes[2].legend()

    plt.tight_layout()
    plt.savefig('perf_analysis.png', dpi=120)
    plt.show()
    print("Plot saved to perf_analysis.png")

    # Target pass / fail
    avg_fps = np.mean(perf.fps_log)
    p95_e2e = np.percentile(perf.e2e_ms, 95)
    print(f"\n{'PASS' if avg_fps >= 15 else 'FAIL'}: avg FPS = {avg_fps:.1f}  (target ≥ 15)")
    print(f"{'PASS' if p95_e2e < 200 else 'FAIL'}: p95 latency = {p95_e2e:.1f} ms  (target < 200 ms)")
else:
    print("No frames recorded — run the main loop first.")
